# 01. Подготовка данных

Вариант 05. Одна строка исходной выгрузки соответствует одному объявлению об аренде.
Целевая переменная – месячная арендная плата `rent_rub` в рублях.

## Пути проекта

Исходные файлы лежат в `data/raw/`. Корень проекта определяем из рабочего каталога ядра, чтобы запуск из `notebooks/` тоже находил пакет `modules`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "modules").is_dir():
    raise RuntimeError("Запустите ноутбук из корня проекта или папки notebooks/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"


## Загрузка и проверка формата CSV

Как в первой лабораторной и практикуме 2, сначала читаем пять строк через `pd.read_csv`. Проверяем кодировку, разделитель полей и десятичный знак по названиям районов, площади и заголовкам.

In [2]:
import pandas as pd

RAW_CSV_PATH = RAW_DATA_DIR / "rent_v05.csv"
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ","
CSV_DECIMAL = "."

sample = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    sep=CSV_SEPARATOR,
    decimal=CSV_DECIMAL,
    nrows=5,
)

assert pd.api.types.is_numeric_dtype(sample["total_area"]), "Площадь не прочиталась как число."

print(f"Кодировка – {CSV_ENCODING.upper()}")
print(f"Разделитель полей – {CSV_SEPARATOR!r}, десятичный знак – {CSV_DECIMAL!r}")
print(f"Пример категории – {sample.loc[0, 'district']}")
print(f"Пример дробного числа – {sample.loc[0, 'total_area']}")


Кодировка – UTF-8
Разделитель полей – ',', десятичный знак – '.'
Пример категории – Кировский
Пример дробного числа – 65.6


### Чтение всей выгрузки

Читаем значения как текст и сохраняем пустые поля, чтобы при записи сырого теста сохранить исходное представление значений. Названия и порядок столбцов совпадают с таблицей в описании варианта 05.

In [3]:
raw_df = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    sep=CSV_SEPARATOR,
    dtype=str,
    keep_default_na=False,
)

print("Размер выгрузки –", raw_df.shape)
print("Столбцы –", ", ".join(raw_df.columns))


Размер выгрузки – (12360, 26)
Столбцы – listing_id, publication_date, district, house_type, year_built, floors_total, floor, elevator, rooms, total_area, living_area, kitchen_area, ceiling_height, renovation, furniture, appliances, balcony, parking, metro_distance_m, distance_to_center_km, pets_allowed, seller_type, description_length, photos_count, max_tenants, rent_rub


## Удаление дублей и пустой цели

До разбиения удаляем строки, которые совпадают по всем столбцам, и строки с пустой ценой аренды. Так одна и та же строка не попадёт в разные выборки, а у каждой записи будет фактическая цена для обучения или проверки.

In [4]:
TARGET = "rent_rub"

unique_df = raw_df.drop_duplicates()
split_df = unique_df.loc[unique_df[TARGET].str.strip().ne("")]

print(f"Удалено точных дублей – {len(raw_df) - len(unique_df)}")
print(f"Удалено строк с пустым rent_rub – {len(unique_df) - len(split_df)}")
print(f"Строк для разбиения – {len(split_df)}")


Удалено точных дублей – 180
Удалено строк с пустым rent_rub – 30
Строк для разбиения – 12150


## Разбиение на train, validation и test

Как в практикуме 5, сначала откладываем 20 % строк в тест. Из оставшихся строк выделяем 20 % на валидацию. В обоих вызовах используем один `random_state`, чтобы при повторном запуске получались те же выборки.

Сырой тест сохраняем в `data/interim/rent_v05_test.csv` в UTF-8 с разделителем-запятой. Для дальнейшего разведочного анализа и расчёта параметров предобработки используем только train, как в лекции 5.1 об утечке данных.

In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.20

development_df, test_df = train_test_split(
    split_df, test_size=TEST_FRACTION, random_state=RANDOM_STATE
)
train_df, validation_df = train_test_split(
    development_df, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE
)

assert train_df.index.intersection(validation_df.index).empty
assert train_df.index.intersection(test_df.index).empty
assert validation_df.index.intersection(test_df.index).empty
assert len(train_df) + len(validation_df) + len(test_df) == len(split_df)

test_path = PROJECT_ROOT / "data" / "interim" / "rent_v05_test.csv"
test_path.parent.mkdir(parents=True, exist_ok=True)
test_df.sort_index().to_csv(
    test_path, encoding=CSV_ENCODING, sep=CSV_SEPARATOR, index=False, lineterminator="\n"
)

print(f"Строк в train – {len(train_df)}")
print(f"Строк в validation – {len(validation_df)}")
print(f"Строк в test – {len(test_df)}")
print("Выборки не пересекаются.")
print(f"Сырой тест сохранён в {test_path.relative_to(PROJECT_ROOT)}.")


Строк в train – 7776
Строк в validation – 1944
Строк в test – 2430
Выборки не пересекаются.
Сырой тест сохранён в data/interim/rent_v05_test.csv.
